# DF20 Mushroom Species Results

This notebook analyzes saved training outputs from the DF20 mushroom experiment. It is designed to work even if only some models have finished.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Image, display
from sklearn.metrics import confusion_matrix

sns.set_theme(style='whitegrid', context='notebook')

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'df20_species_project'
FIGURES_DIR = OUTPUT_DIR / 'figures'
TABLES_DIR = OUTPUT_DIR / 'tables'
PREDICTIONS_DIR = OUTPUT_DIR / 'predictions'

def first_existing(*paths):
    for path in paths:
        if path.exists():
            return path
    return paths[0]

OUTPUT_DIR

## Dataset Split and Balance

In [ ]:
split_summary_path = TABLES_DIR / 'split_summary.csv'
species_distribution_path = TABLES_DIR / 'species_distribution.csv'

if split_summary_path.exists():
    split_summary = pd.read_csv(split_summary_path)
    display(split_summary)
else:
    print('split_summary.csv will appear after the next training run.')

if species_distribution_path.exists():
    species_distribution = pd.read_csv(species_distribution_path)
    display(species_distribution.head(20))
    plt.figure(figsize=(10, 4.5))
    sns.histplot(species_distribution['total_count'], bins=30)
    plt.title('Species image-count distribution')
    plt.xlabel('Images per selected species')
    plt.ylabel('Number of species')
    plt.tight_layout()
    plt.show()
else:
    print('species_distribution.csv will appear after the next training run.')

## Summary Table

In [ ]:
results_path = first_existing(TABLES_DIR / 'results.csv', OUTPUT_DIR / 'results.csv')
if not results_path.exists():
    raise FileNotFoundError(f'No results.csv found at {results_path}')

results = pd.read_csv(results_path)
metric_cols = [
    'display_name', 'test_accuracy', 'test_top3_accuracy', 'test_f1_macro',
    'test_balanced_accuracy', 'risk_accuracy', 'dangerous_error_rate', 'risk_coverage'
]
summary = results[metric_cols].sort_values('test_f1_macro', ascending=False)
styled = summary.style.format({
    'test_accuracy': '{:.1%}',
    'test_top3_accuracy': '{:.1%}',
    'test_f1_macro': '{:.1%}',
    'test_balanced_accuracy': '{:.1%}',
    'risk_accuracy': '{:.1%}',
    'dangerous_error_rate': '{:.1%}',
    'risk_coverage': '{:.1%}',
})
styled

## Model Comparison Graph

In [ ]:
plot_df = summary.melt(
    id_vars='display_name',
    value_vars=['test_accuracy', 'test_top3_accuracy', 'test_f1_macro', 'dangerous_error_rate'],
    var_name='metric',
    value_name='value',
)
metric_labels = {
    'test_accuracy': 'Top-1 accuracy',
    'test_top3_accuracy': 'Top-3 accuracy',
    'test_f1_macro': 'Macro F1',
    'dangerous_error_rate': 'Dangerous error',
}
plot_df['metric'] = plot_df['metric'].map(metric_labels)

plt.figure(figsize=(11, 5.5))
ax = sns.barplot(data=plot_df, x='display_name', y='value', hue='metric')
ax.set_ylim(0, 1)
ax.set_xlabel('')
ax.set_ylabel('Score')
ax.set_title('DF20 mushroom species model comparison')
ax.tick_params(axis='x', rotation=15)
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=8, padding=2)
plt.legend(title='Metric', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Training Curves

In [ ]:
history_paths = sorted(FIGURES_DIR.glob('*_history.png')) or sorted(OUTPUT_DIR.glob('*_history.png'))
for image_path in history_paths:
    print(image_path.name)
    display(Image(filename=str(image_path)))

## Risk Confusion Matrices

In [ ]:
risk_paths = sorted(FIGURES_DIR.glob('*_risk_confusion.png')) or sorted(OUTPUT_DIR.glob('*_risk_confusion.png'))
for image_path in risk_paths:
    print(image_path.name)
    display(Image(filename=str(image_path)))

## Species Confusion Matrix for One Model

A full 100-class confusion matrix is hard to read, so this cell shows the most common true species for the selected model.

In [ ]:
MODEL_FOR_CONFUSION = results.sort_values('test_f1_macro', ascending=False).iloc[0]['model_name']
TOP_N_SPECIES = 20

pred_path = first_existing(PREDICTIONS_DIR / f'{MODEL_FOR_CONFUSION}_predictions.csv', OUTPUT_DIR / f'{MODEL_FOR_CONFUSION}_predictions.csv')
predictions = pd.read_csv(pred_path)
top_species = predictions['true_species_name'].value_counts().head(TOP_N_SPECIES).index.tolist()
filtered = predictions[
    predictions['true_species_name'].isin(top_species)
    & predictions['pred_species_name'].isin(top_species)
]
matrix = confusion_matrix(
    filtered['true_species_name'],
    filtered['pred_species_name'],
    labels=top_species,
)

plt.figure(figsize=(12, 10))
sns.heatmap(matrix, cmap='Blues', xticklabels=top_species, yticklabels=top_species, square=True)
plt.title(f'Species confusion matrix: {MODEL_FOR_CONFUSION} (top {TOP_N_SPECIES} true species)')
plt.xlabel('Predicted species')
plt.ylabel('True species')
plt.xticks(rotation=70, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Most Common Species Confusions

In [ ]:
confusion_tables = []
top_confusion_paths = sorted(TABLES_DIR.glob('*_top_confusions.csv')) or sorted(OUTPUT_DIR.glob('*_top_confusions.csv'))
for path in top_confusion_paths:
    model_name = path.stem.replace('_top_confusions', '')
    table = pd.read_csv(path).head(10)
    table.insert(0, 'model_name', model_name)
    confusion_tables.append(table)

if confusion_tables:
    display(pd.concat(confusion_tables, ignore_index=True))
else:
    print('No top-confusion files found yet.')

## Abstention Analysis

In [ ]:
abstention_tables = []
abstention_paths = sorted(TABLES_DIR.glob('*_abstention.csv')) or sorted(OUTPUT_DIR.glob('*_abstention.csv'))
for path in abstention_paths:
    model_name = path.stem.replace('_abstention', '')
    table = pd.read_csv(path)
    table.insert(0, 'model_name', model_name)
    abstention_tables.append(table)

if abstention_tables:
    abstention = pd.concat(abstention_tables, ignore_index=True)
    display(abstention.style.format({
        'threshold': '{:.1f}',
        'coverage': '{:.1%}',
        'retained_accuracy': '{:.1%}',
        'retained_macro_f1': '{:.1%}',
        'retained_risk_accuracy': '{:.1%}',
        'retained_dangerous_error_rate': '{:.1%}',
    }))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
    sns.lineplot(data=abstention, x='threshold', y='coverage', hue='model_name', marker='o', ax=axes[0])
    sns.lineplot(data=abstention, x='threshold', y='retained_dangerous_error_rate', hue='model_name', marker='o', ax=axes[1])
    axes[0].set_title('Coverage after abstention')
    axes[1].set_title('Dangerous error after abstention')
    axes[0].set_ylim(0, 1)
    axes[1].set_ylim(0, 1)
    for axis in axes:
        axis.set_xlabel('Confidence threshold')
    plt.tight_layout()
    plt.show()
else:
    print('No abstention files found yet.')

## Interpretation Notes

- Compare models using macro F1 and top-3 accuracy, not only top-1 accuracy.
- A high train score with lower test score is expected in fine-grained mushroom recognition and should be discussed as overfitting risk.
- Dangerous error rate is the most safety-relevant metric: lower is better.
- Abstention is useful if it reduces dangerous errors while keeping acceptable coverage.